# 팔도무록 추궁 답변 LLM MVP

마을 사람이 무당의 추궁에 답하는 대사를 로컬 LLM(llama.cpp, GGUF)으로 만들고 측정합니다.
사실은 게임 엔진이 정한 문장(`fact_sentence`)을 그대로 쓰고, LLM은 말투와 감정만 입힙니다(기획서 7.2).
생성문은 검증기를 통과해야 쓰이고, 실패하면 사실 문장이 든 폴백 문장을 씁니다.

**실행 순서**: 위에서부터 한 칸씩(또는 `런타임 → 모두 실행`) 실행합니다. 무료 CPU 런타임 기준 목표는 전체 40분 이하입니다.

| 단계 | 내용 |
|---|---|
| ① | 입력 칸: 드라이브 파일 ID, 일관성 건수·반복 수, 로컬 경로 |
| ② | llama-cpp-python 설치(미리 빌드된 휠) |
| ③ | 구글 드라이브에서 모델 GGUF와 코드 묶음 zip 받기, sha256 출력, **생성 설정 표시** |
| ④ | 모델 로드, 실행 환경 출력, 예시 3건 추론 |
| ⑤ | 지표 표: 성공률(100건), 지연 시간, 일관성 |
| ⑥ | 직접 질문 넣기 |

지표 계산은 zip 안 `llm/mvp_metrics.py`가 하고, 이 노트북은 설치·다운로드·llama.cpp 호출·표 출력만 합니다.

## ① 입력 칸

- `MODEL_FILE_ID`, `ZIP_FILE_ID`: 공유된 드라이브 파일의 ID(링크의 `/d/`와 `/view` 사이 문자열).
- `CONSISTENCY_N`, `CONSISTENCY_K`: 일관성 측정 건수(평가 세트 앞에서부터)와 사례마다 반복 수(시드 1~K). 기본 20건 × 5회. 시간이 모자라면 줄일 수 있고, 줄이면 표에 표시됩니다.
- `LOCAL_MODEL_PATH`, `LOCAL_ZIP_PATH`: 이미 받아 둔 파일 경로. 값이 있으면 ③에서 다운로드를 건너뜁니다.
- 생성 설정(온도·top_p·top_k·최대 토큰·컨텍스트 길이·스레드 수)은 zip 안 `llm/prompt.py`의 `GEN_SETTINGS`를 그대로 씁니다. zip을 받은 직후(③) 표로 표시합니다. 스레드 수만 `min(GEN_SETTINGS의 n_threads, CPU 수)`로 정합니다.

In [ ]:
import time
T_START = time.time()   # 전체 실행 시간 측정 시작

MODEL_FILE_ID = ""      # 모델 GGUF 드라이브 파일 ID
ZIP_FILE_ID = ""        # 코드 묶음 zip 드라이브 파일 ID
CONSISTENCY_N = 20      # 일관성 측정 건수(기본 20)
CONSISTENCY_K = 5       # 사례마다 반복 수 = 시드 1~K(기본 5)
LOCAL_MODEL_PATH = ""   # 로컬 GGUF 경로(값이 있으면 다운로드 건너뜀)
LOCAL_ZIP_PATH = ""     # 로컬 zip 경로(값이 있으면 다운로드 건너뜀)

## ② llama-cpp-python 설치

- GPU가 없으면(무료 CPU 런타임) 미리 빌드된 CPU 휠을 받습니다. 소스 빌드는 하지 않습니다(`--only-binary=:all:`).
- GPU가 있으면 CUDA 휠을 먼저 시도하고, 없으면 CPU 휠로 설치합니다. GPU 휠이면 ④에서 모든 층을 GPU에 올립니다(`n_gpu_layers=-1`).
- 같은 버전이 이미 설치돼 있으면 건너뜁니다.

In [ ]:
import importlib.metadata
import shutil
import subprocess
import sys

LLAMA_CPP_VERSION = "0.3.36"   # 저장소 requirements/llm.txt와 같은 버전
CPU_INDEX = "https://abetlen.github.io/llama-cpp-python/whl/cpu"
GPU_INDEX = "https://abetlen.github.io/llama-cpp-python/whl/cu124"


def gpu_name():
    # nvidia-smi가 있으면 GPU 이름, 없으면 None
    if not shutil.which("nvidia-smi"):
        return None
    out = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True)
    return out.stdout.strip().splitlines()[0] if out.returncode == 0 and out.stdout.strip() else None


def pip_install(index):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "--extra-index-url", index,
           f"llama-cpp-python=={LLAMA_CPP_VERSION}"]
    print(" ".join(cmd))
    return subprocess.run(cmd).returncode == 0


GPU = gpu_name()
try:
    installed = importlib.metadata.version("llama-cpp-python")
except importlib.metadata.PackageNotFoundError:
    installed = None

if installed == LLAMA_CPP_VERSION:
    print(f"llama-cpp-python {installed} 이미 설치됨, 건너뜀")
else:
    ok = GPU is not None and pip_install(GPU_INDEX)
    if GPU is not None and not ok:
        print("GPU 휠 설치 실패 → CPU 휠로 설치")
    if not ok and not pip_install(CPU_INDEX):
        raise RuntimeError(f"llama-cpp-python {LLAMA_CPP_VERSION} 휠 설치 실패. 런타임 Python 버전과 네트워크를 확인하세요.")

import llama_cpp
print("llama-cpp-python", llama_cpp.__version__, "| GPU:", GPU or "없음")

## ③ 모델 GGUF와 코드 묶음 zip 받기

공개 링크("링크가 있는 누구나" 보기) 파일을 ID로 받습니다(`gdown`). 큰 파일의 바이러스 검사 확인 단계는 `gdown`이 처리합니다.
받은 파일의 sha256을 출력합니다. zip을 풀어 `llm` 코드를 불러오고 생성 설정을 표시합니다.

In [ ]:
import hashlib
import os
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
WORK = Path("/content/paldo_mvp") if IN_COLAB else Path.cwd() / "paldo_mvp_work"
WORK.mkdir(parents=True, exist_ok=True)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def check_file(path, kind):
    # 받은 파일이 기대한 형식인지 본다. 권한이 없으면 드라이브가 HTML 안내 페이지를 돌려준다
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return f"{kind} 파일이 없거나 비어 있음: {path}"
    with open(path, "rb") as f:
        head = f.read(4)
    if kind == "model" and head != b"GGUF":
        return f"GGUF 파일이 아님(앞 4바이트 {head!r}). 공유 범위가 '링크가 있는 누구나'인지, ID가 맞는지 확인하세요."
    if kind == "zip" and not zipfile.is_zipfile(path):
        return "zip 파일이 아님. 공유 범위가 '링크가 있는 누구나'인지, ID가 맞는지 확인하세요."
    return None


def fetch(local_path, file_id, name, kind):
    if local_path:
        print(f"{kind}: 로컬 경로 사용, 다운로드 건너뜀")
        path = Path(local_path)
    else:
        if not file_id:
            raise ValueError(f"{kind}: ① 입력 칸에 파일 ID 또는 로컬 경로를 넣으세요.")
        try:
            import gdown
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
            import gdown
        path = WORK / name
        marker = WORK / f"{name}.source"   # 받은 파일의 ID. ID가 바뀌면 다시 받는다
        same = marker.is_file() and marker.read_text() == file_id
        if not (same and check_file(path, kind) is None):   # 같은 ID를 같은 런타임에서 다시 실행하면 재사용
            marker.unlink(missing_ok=True)
            try:
                result = gdown.download(id=file_id, output=str(path), quiet=False)
            except Exception as exc:
                raise RuntimeError(f"{kind} 다운로드 실패: {exc}. 공유 범위('링크가 있는 누구나' 보기)와 파일 ID를 확인하고, "
                                   "드라이브 하루 다운로드 한도에 걸렸으면 잠시 뒤 다시 실행하세요.") from None
            if result is None:
                raise RuntimeError(f"{kind} 다운로드 실패. 공유 범위와 파일 ID를 확인하세요.")
            if check_file(path, kind) is None:
                marker.write_text(file_id)
    problem = check_file(path, kind)
    if problem:
        raise RuntimeError(problem)
    print(f"{kind}: {path.name}  {path.stat().st_size:,} 바이트  sha256 {sha256(path)}")
    return path


MODEL_PATH = fetch(LOCAL_MODEL_PATH, MODEL_FILE_ID, "model.gguf", "model")
ZIP_PATH = fetch(LOCAL_ZIP_PATH, ZIP_FILE_ID, "paldo_llm_mvp.zip", "zip")

BUNDLE_DIR = WORK / "bundle"
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(BUNDLE_DIR)
    print("zip 내용:", ", ".join(zf.namelist()))

# zip 안 llm 코드만 쓰도록 패키지 위치를 BUNDLE_DIR/llm 하나로 고정한다. 저장소 안에서 실행해 경로에 다른 llm
# 패키지가 보이거나, 다른 zip으로 다시 실행해 이전 모듈이 남아 있어도 섞이지 않게 한다
import importlib.machinery
import importlib.util

for _name in [m for m in sys.modules if m == "llm" or m.startswith("llm.")]:
    del sys.modules[_name]
_spec = importlib.machinery.ModuleSpec("llm", None, is_package=True)
_spec.submodule_search_locations = [str(BUNDLE_DIR / "llm")]
sys.modules["llm"] = importlib.util.module_from_spec(_spec)

from llm import mvp_metrics, prompt, validator   # zip 안 코드

assert Path(mvp_metrics.__file__).resolve().is_relative_to(BUNDLE_DIR.resolve()), mvp_metrics.__file__
print("llm 코드 위치:", Path(mvp_metrics.__file__).parent)

In [ ]:
import json
from IPython.display import Markdown, display

EVAL = json.loads((BUNDLE_DIR / "interrogation_eval.json").read_text(encoding="utf-8"))
CASES = EVAL["cases"]
LEXICON = validator.Lexicon.from_json(EVAL["lexicon"])
GEN = dict(prompt.GEN_SETTINGS)
N_THREADS = min(GEN["n_threads"], os.cpu_count() or 1)


def table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(str(x) for x in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


print("모델 파일(기대):", prompt.MODEL_FILE, "| 프롬프트 버전:", prompt.PROMPT_VERSION,
      "| 평가 세트:", EVAL["eval_version"], f"{len(CASES)}건")
table(["생성 설정", "값"], [
    ["temperature", GEN["temperature"]], ["top_p", GEN["top_p"]], ["top_k", GEN["top_k"]],
    ["max_tokens", GEN["max_tokens"]], ["n_ctx", GEN["n_ctx"]],
    ["n_threads", f"GEN_SETTINGS {GEN['n_threads']} → 실제 {N_THREADS} (min(설정, CPU 수 {os.cpu_count()}))"],
])

## ④ 모델 로드와 예시 3건

채팅 템플릿은 GGUF에 든 것을 직접 그려 사고(thinking) 모드를 끕니다. 요청마다 llama.cpp 성능 카운터로 프롬프트 처리·생성 시간을 나눠 잽니다(저장소 측정과 같은 방식).

In [ ]:
import contextlib
import io
import platform
from datetime import datetime

from jinja2 import nodes
from jinja2.ext import Extension
from jinja2.sandbox import ImmutableSandboxedEnvironment


class _Generation(Extension):
    # 일부 채팅 템플릿의 {% generation %} 블록을 본문 그대로 출력한다
    tags = {"generation"}

    def parse(self, parser):
        lineno = next(parser.stream).lineno
        body = parser.parse_statements(("name:endgeneration",), drop_needle=True)
        return nodes.Scope(body, lineno=lineno)


def _raise(msg):
    raise RuntimeError(msg)


N_GPU_LAYERS = -1 if (GPU and llama_cpp.llama_supports_gpu_offload()) else 0
t0 = time.perf_counter()
with contextlib.redirect_stderr(io.StringIO()):
    LLM = llama_cpp.Llama(model_path=str(MODEL_PATH), n_threads=N_THREADS, n_threads_batch=N_THREADS,
                          n_ctx=GEN["n_ctx"], n_gpu_layers=N_GPU_LAYERS, seed=1, verbose=True)
LLM.verbose = False
print(f"모델 로드 {time.perf_counter() - t0:.1f} s")
_env = ImmutableSandboxedEnvironment(trim_blocks=True, lstrip_blocks=True, extensions=[_Generation])
_env.globals["raise_exception"] = _raise
_env.globals["strftime_now"] = lambda fmt: datetime.now().strftime(fmt)
TEMPLATE = _env.from_string(LLM.metadata["tokenizer.chat_template"])


def generate(messages, *, seed):
    # mvp_metrics.run_eval에 넘기는 생성 함수. 앞 요청의 KV 캐시는 재사용하지 않는다
    text = TEMPLATE.render(messages=messages, add_generation_prompt=True, enable_thinking=False,
                           bos_token="", eos_token="")
    LLM.reset()
    ctx = LLM._ctx.ctx
    llama_cpp.llama_perf_context_reset(ctx)
    out = LLM.create_completion(text, max_tokens=GEN["max_tokens"], temperature=GEN["temperature"],
                                top_p=GEN["top_p"], top_k=GEN["top_k"], stop=prompt.STOP, seed=seed)
    perf = llama_cpp.llama_perf_context(ctx)
    return {"text": out["choices"][0]["text"], "prompt_ms": perf.t_p_eval_ms, "gen_ms": perf.t_eval_ms,
            "prompt_tokens": int(perf.n_p_eval), "gen_tokens": int(perf.n_eval)}


def cpu_name():
    try:
        if sys.platform.startswith("linux"):
            for line in Path("/proc/cpuinfo").read_text().splitlines():
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
        if sys.platform == "win32":
            out = subprocess.run(["powershell", "-NoProfile", "-Command", "(Get-CimInstance Win32_Processor).Name"],
                                 capture_output=True, text=True)
            if out.stdout.strip():
                return out.stdout.strip().splitlines()[0]
    except Exception:
        pass
    return platform.processor() or "알 수 없음"


ENV = {"cpu": cpu_name(), "cpu_count": os.cpu_count(), "gpu": GPU or "없음", "n_threads": N_THREADS,
       "n_gpu_layers": N_GPU_LAYERS, "llama_cpp": llama_cpp.__version__, "python": platform.python_version(),
       "runtime": "colab" if IN_COLAB else "local", "model_file": MODEL_PATH.name}
table(["실행 환경", "값"], [["CPU", ENV["cpu"]], ["CPU 수(논리)", ENV["cpu_count"]], ["GPU", ENV["gpu"]],
                         ["스레드 수", ENV["n_threads"]], ["n_gpu_layers", ENV["n_gpu_layers"]],
                         ["llama-cpp-python", ENV["llama_cpp"]], ["Python", ENV["python"]]])

generate(prompt.build_messages(CASES[0]["request"]), seed=0)   # 예열 1건(집계 제외)


def show(records, cases):
    rows = []
    for rec, case in zip(records, cases):
        req = case["request"]
        rows.append([rec["case_id"], req["reaction"], req["fact_sentence"], rec["generated"] or "",
                     "통과" if rec["status"] == "done" else f"{rec['status']} {','.join(rec['reasons'])}",
                     rec["text"], f"{rec['total_ms']:,.0f}"])
    table(["사례", "반응", "사실 문장", "생성문", "판정", "화면 문장", "지연(ms)"], rows)


kinds_seen, EXAMPLES = set(), []
for case in CASES:   # 종류가 다른 앞 사례 3건
    if case["kind"] not in kinds_seen and len(EXAMPLES) < 3:
        kinds_seen.add(case["kind"])
        EXAMPLES.append(case)
show(mvp_metrics.run_eval(EXAMPLES, generate, seed=0, lexicon=LEXICON), EXAMPLES)

## ⑤ 지표

- **성공률**: 평가 세트 100건을 시드 0으로 1회씩 생성합니다. 성공 = 검증기를 통과하고 폴백 없이 나온 답입니다. 시간 초과는 요청 합계가 `deadline_ms`(15초)를 넘은 경우로, 사후에 세어 폴백으로 처리합니다.
- **지연 시간**: 요청마다 프롬프트 처리·생성·검증 시간과 합계의 p50·p95(최근접 순위), 생성 토큰/초 평균. 15초 기준(NFR-LLM-001)은 코랩 사양이 달라 참고로만 표시합니다.
- **일관성**: 앞 `CONSISTENCY_N`건을 시드 1~`CONSISTENCY_K`로 생성해 ① 판정 일치 ② 사실 요소 보존 ③ 문자 유사도(`difflib.SequenceMatcher`, 모든 쌍 평균)를 구합니다.
- 정의와 계산은 모두 zip 안 `llm/mvp_metrics.py`입니다.

In [ ]:
t0 = time.perf_counter()
RECORDS = mvp_metrics.run_eval(CASES, generate, seed=0, lexicon=LEXICON)
EVAL_S = time.perf_counter() - t0
SM = mvp_metrics.success_metrics(RECORDS)
LM = mvp_metrics.latency_metrics(RECORDS)
v = lambda m, k: m[k]["value"] if k in m else "-"   # noqa: E731  (생성이 모두 실패하면 단계별 값이 없다)

print(f"성공률 측정 {len(RECORDS)}건, {EVAL_S / 60:.1f}분")
rows = [["성공률", f"{v(SM, 'success_rate_pct')} % ({v(SM, 'success_count')}/{v(SM, 'n_requests')}건)"]]
rows += [[f"검증 실패 사유 `{code}`", f"{v(SM, f'reason_{code}_count')}건 ({v(SM, f'reason_{code}_pct')} %)"]
         for code in validator.REASONS]
rows += [["생성 오류", f"{v(SM, 'error_count')}건"], ["시간 초과", f"{v(SM, 'timeout_count')}건"],
         ["폴백 문장이 사실 문장 포함", f"{v(SM, 'fallback_has_fact_pct')} %"]]
table([f"성공률(시드 0, {len(RECORDS)}건)", "값"], rows)

ref = "이하" if v(LM, "total_p95_ms") <= 15000 else "초과"
table(["지연 시간", "p50 (ms)", "p95 (ms)"], [
    ["프롬프트 처리", v(LM, "prompt_p50_ms"), v(LM, "prompt_p95_ms")],
    ["생성", v(LM, "gen_p50_ms"), v(LM, "gen_p95_ms")],
    ["검증", v(LM, "validate_p50_ms"), v(LM, "validate_p95_ms")],
    ["합계", v(LM, "total_p50_ms"), v(LM, "total_p95_ms")],
])
print(f"생성 토큰/초 평균 {v(LM, 'gen_tps_mean')} tok/s, 평균 프롬프트 {v(LM, 'prompt_tokens_mean')} tok, "
      f"평균 생성 {v(LM, 'gen_tokens_mean')} tok")
print(f"참고(NFR-LLM-001, 코랩 사양이 달라 판정 아님): 합계 p95 {v(LM, 'total_p95_ms'):,} ms → 15,000 ms {ref}")

In [ ]:
REDUCED = CONSISTENCY_N < 20 or CONSISTENCY_K < 5
t0 = time.perf_counter()
CM, RUNS = mvp_metrics.run_consistency(CASES[:CONSISTENCY_N], generate, seeds=list(range(1, CONSISTENCY_K + 1)),
                                       lexicon=LEXICON)
CONS_S = time.perf_counter() - t0
label = f"앞 {v(CM, 'consistency_cases')}건 × 시드 1~{v(CM, 'consistency_k')}"
if REDUCED:
    label += " **(기본 20건 × 5회에서 줄임)**"
print(f"일관성 측정 {CONS_S / 60:.1f}분")
table([f"일관성({label})", "값"], [
    ["① 판정 일치", f"{v(CM, 'verdict_agree_pct')} %"],
    ["② 사실 요소 보존(모든 회차)", f"{v(CM, 'facts_kept_pct')} %"],
    ["③ 문자 유사도 평균", v(CM, "similarity_mean")],
])

In [ ]:
ELAPSED_S = time.time() - T_START
print(f"전체 실행 시간(① 칸부터 여기까지): {ELAPSED_S / 60:.1f}분 (목표 40분 이하)")
RESULTS = {"env": ENV, "gen_settings": GEN, "n_threads_used": N_THREADS, "prompt_version": prompt.PROMPT_VERSION,
           "eval_version": EVAL["eval_version"], "consistency_n": CONSISTENCY_N, "consistency_k": CONSISTENCY_K,
           "reduced": REDUCED, "elapsed_s": round(ELAPSED_S, 1), "eval_s": round(EVAL_S, 1),
           "consistency_s": round(CONS_S, 1), "success": SM, "latency": LM, "consistency": CM,
           "verdicts": {r["case_id"]: r["status"] for r in RECORDS}}
RESULTS_PATH = WORK / "paldo_llm_mvp_results.json"
RESULTS_PATH.write_text(json.dumps(RESULTS, ensure_ascii=False, indent=1), encoding="utf-8")
print("결과 저장:", RESULTS_PATH)

## ⑥ 직접 질문 넣기

인물, 사실 문장, 무당의 질문을 넣고 실행하면 답변, 검증 결과, 지연 시간을 보여 줍니다.

- 게임에서는 엔진이 질문에 대한 **사실 문장**과 **반응 의도**(`explain` 해명, `evade` 얼버무림, `counter` 되물음, `confess` 고백)를 정하고, LLM은 그 사실을 바꾸지 않고 말투만 입힙니다. 그래서 프롬프트(`interrogate-v1`)에는 질문 문장 대신 사실 문장과 반응 의도가 들어갑니다. 질문은 화면 표시용이고 비워 둬도 실행됩니다.
- 실제 게임에서는 질문을 해석기(기획서 9.1)와 엔진이 반응 의도·사실 문장으로 바꿉니다. 이 노트북은 그 단계를 직접 고르게 한 것입니다.
- 사실 문장에 없는 사람·장소·시간을 생성문이 말하면 검증에서 걸리고 폴백 문장을 씁니다.

In [ ]:
PERSON_NAME = "뱃사공"                    # 인물 이름
PERSON_STYLE = "투박한 사투리"          # 말투
QUESTION = "2일째 밤에 어디 있었소?"         # 무당의 질문(화면 표시용, 비워도 됨)
FACT = "2일째 밤 한밤엔 나루터에 있었소."     # 엔진이 정한 사실 문장(답에 그대로 들어가야 함)
REACTION = "explain"                    # explain | evade | counter | confess

req = {"job_id": "custom", "persona": f"{PERSON_NAME}. 말투: {PERSON_STYLE}", "fact_sentence": FACT,
       "fallback_text": FACT, "reaction": REACTION, "allowed_names": [PERSON_NAME], "deadline_ms": 15000}
rec = mvp_metrics.run_eval([{"case_id": "custom", "request": req}], generate, seed=0, lexicon=LEXICON)[0]
if QUESTION:
    print("무당:", QUESTION)
print(f"{PERSON_NAME}:", rec["text"])
print("판정:", "통과" if rec["status"] == "done" else f"{rec['status']} (폴백) {', '.join(rec['reasons'])}")
if rec["details"]:
    print("검증 상세:", "; ".join(rec["details"]))
print("생성문(검증 전):", rec["generated"])
print(f"지연: 합계 {rec['total_ms']:,.0f} ms = 프롬프트 처리 {rec['prompt_ms']:,.0f} + 생성 {rec['gen_ms']:,.0f} "
      f"+ 검증 {rec['validate_ms']:.2f} ms (그 밖 템플릿 처리 등)")